In [25]:
import dlt
import requests
import time

from datetime import datetime, timezone
from uuid import uuid4

from dlt.destinations import filesystem
from dlt.destinations.adapters import iceberg_adapter, iceberg_partition

In [26]:
SOURCE_SYSTEM = "openlibrary"
INGESTION_TYPE = "full"

INGESTED_AT = datetime.now(timezone.utc)
RUN_ID = str(uuid4())

BRONZE_BUCKET_URL = "s3://book-library-lakehouse/bronze"

destination = filesystem(
    bucket_url=BRONZE_BUCKET_URL
)

iceberg_partition.day("_ingested_at")

PartitionSpec(source_column='_ingested_at', transform='day', param_value=None, partition_field=None)

In [27]:
QUERY = "title:python"
API_URL = "https://openlibrary.org/search.json"
PAGE_SIZE = 100
HEADERS = {
    "User-Agent": "BookLibraryDataPipeline/1.0"
}

@dlt.resource(
    name="books",
    write_disposition="replace"
)

def openlibrary_books():

    offset = 0
    total_found = None

    with requests.Session() as session:

        while total_found is None or offset < total_found:

            params = {
                "q": QUERY,
                "fields": (
                    "key,title,author_name,author_key,"
                    "first_publish_year,edition_count,isbn,language"
                ),
                "limit": PAGE_SIZE,
                "offset": offset
            }

            response = session.get(
                API_URL,
                params=params,
                headers=HEADERS,
                timeout=60
            )

            response.raise_for_status()

            data = response.json()

            if total_found is None:
                total_found = int(data.get("numFound", 0))

                print("Najdenih knjig:", total_found)

                if total_found == 0:
                    raise RuntimeError(
                        "API ni vrnil nobene knjige. "
                        "Preveri query. Replace je bil prekinjen."
                    )

            docs = data.get("docs", [])

            if not docs:
                break

            for book in docs:

                item = dict(book)

                author_names = item.pop("author_name", []) or []
                author_keys = item.pop("author_key", []) or []

                item["authors"] = "; ".join(author_names)
                item["author_ids"] = "; ".join(author_keys)

                item["openlibrary_id"] = item.get("key")

                # metadata
                item["_source_system"] = SOURCE_SYSTEM
                item["_source_resource"] = "books"
                item["_ingestion_type"] = INGESTION_TYPE
                item["_ingested_at"] = INGESTED_AT
                item["_load_id"] = RUN_ID

                yield item

            offset += len(docs)

            print(
                f"Prenesenih {offset} od {total_found} knjig."
            )

            if offset < total_found:
                time.sleep(1.1)

    if offset < total_found:
        raise RuntimeError(
            f"Nalaganje ni bilo popolno: prenesenih {offset} "
            f"od {total_found} rezultatov."
        )

In [28]:
books_resource = openlibrary_books()

iceberg_adapter(
    books_resource,
    partition=[
        iceberg_partition.day("_ingested_at")
    ]
)

<@dlt.resource(name='books', section='__main__', write_disposition='replace')>

In [29]:
pipeline = dlt.pipeline(
    pipeline_name="openlibrary_to_lakehouse",
    destination=destination,
    dataset_name="openlibrary"
)

In [30]:
load_info = pipeline.run(
    books_resource,
    write_disposition="replace",
    table_format="iceberg"
)

print("Open Library - Iceberg BRONZE ... DONE")
print(load_info)
print("RUN ID:", RUN_ID)
print("INGESTED AT:", INGESTED_AT)

Najdenih knjig: 5856
Prenesenih 100 od 5856 knjig.
Prenesenih 200 od 5856 knjig.
Prenesenih 300 od 5856 knjig.
Prenesenih 400 od 5856 knjig.
Prenesenih 500 od 5856 knjig.
Prenesenih 600 od 5856 knjig.
Prenesenih 700 od 5856 knjig.
Prenesenih 800 od 5856 knjig.
Prenesenih 900 od 5856 knjig.
Prenesenih 1000 od 5856 knjig.
Prenesenih 1100 od 5856 knjig.
Prenesenih 1200 od 5856 knjig.
Prenesenih 1300 od 5856 knjig.
Prenesenih 1400 od 5856 knjig.
Prenesenih 1500 od 5856 knjig.
Prenesenih 1600 od 5856 knjig.
Prenesenih 1700 od 5856 knjig.
Prenesenih 1800 od 5856 knjig.
Prenesenih 1900 od 5856 knjig.
Prenesenih 2000 od 5856 knjig.
Prenesenih 2100 od 5856 knjig.
Prenesenih 2200 od 5856 knjig.
Prenesenih 2300 od 5856 knjig.
Prenesenih 2400 od 5856 knjig.
Prenesenih 2500 od 5856 knjig.
Prenesenih 2600 od 5856 knjig.
Prenesenih 2700 od 5856 knjig.
Prenesenih 2800 od 5856 knjig.
Prenesenih 2900 od 5856 knjig.
Prenesenih 3000 od 5856 knjig.
Prenesenih 3100 od 5856 knjig.
Prenesenih 3200 od 5856 knj